# Earnings Credit Spread Backtest

This notebook backtests a **risk-neutral credit spread strategy** around earnings announcements.

## Strategy Rules

**Entry Criteria:**
- US stock options
- Earnings on **Friday** (BMO or AMC, not during market hours)
- Options expiring the **same week** as earnings
- **Short leg**: ~0.20 delta (OTM)
- **Long leg**: 2 strikes further OTM
- **Credit received >= Risk taken** (risk-neutral)
  - Risk = Strike Width - Credit
  - So: Credit >= Strike Width / 2

**Position Types:**
- **Bull Put Spread**: Sell higher strike put, buy lower strike put (bullish)
- **Bear Call Spread**: Sell lower strike call, buy higher strike call (bearish)

**Backtest Period:** 5 years

## Data Approach
Since free historical options data is not available, we:
1. Fetch historical earnings dates
2. Get historical stock prices and volatility
3. Estimate option prices using Black-Scholes
4. Simulate spread entry and calculate P&L at expiration

In [24]:
# ── Import Libraries ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
from datetime import datetime, timedelta, date
import time
from typing import Optional, Tuple, Dict, List
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set plot style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Configuration

In [67]:
# ── Backtest Configuration ───────────────────────────────────────────────────

# Backtest period
BACKTEST_YEARS = 5
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=BACKTEST_YEARS * 365)

# Strategy parameters
TARGET_DELTA = 0.20           # Short leg delta target
DELTA_TOLERANCE = 0.05        # Accept delta between 0.15 and 0.25
STRIKES_APART = 2             # Long leg is 2 strikes further OTM
RISK_FREE_RATE = 0.03         # Average risk-free rate over period

# Risk-neutral requirement: credit >= risk
# Risk = strike_width - credit
# So: credit >= strike_width - credit => 2*credit >= strike_width
MIN_CREDIT_RATIO = 0.50       # Credit must be >= 50% of strike width

# IV assumptions (will be estimated from historical data)
EARNINGS_IV_MULTIPLIER = 1.5  # IV typically elevated before earnings

# Universe helpers
import urllib.request, json as _json, zipfile, io, re
import xml.etree.ElementTree as ET

_SKIP_STRINGS = {'SPY', 'USD', 'SEDOL', 'ISIN', 'NA', 'ETF', 'BMO', 'AMC', 'OTC'}

def get_sp500_tickers() -> list:
    """
    Fetch S&P 500 constituents from the SPDR SPY ETF holdings XLSX (State Street).
    Parses the shared strings XML directly to avoid openpyxl dependency.
    """
    url = 'https://www.ssga.com/us/en/intermediary/etfs/library-content/products/fund-data/etfs/us/holdings-daily-us-en-spy.xlsx'
    req = urllib.request.Request(url, headers={'User-Agent': 'earnings-backtest/1.0'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        raw = resp.read()

    zf = zipfile.ZipFile(io.BytesIO(raw))
    ss_xml = zf.read('xl/sharedStrings.xml').decode()
    ns = {'ns': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    root = ET.fromstring(ss_xml)
    strings = [
        ''.join(t.text or '' for t in si.findall('.//ns:t', ns))
        for si in root.findall('ns:si', ns)
    ]

    tickers = sorted({
        s.strip() for s in strings
        if re.fullmatch(r'[A-Z]{1,5}', s.strip()) and s.strip() not in _SKIP_STRINGS
    })
    return tickers

def get_all_us_equities() -> list:
    """Fetch all US equity tickers from SEC EDGAR (covers NYSE, NASDAQ, AMEX)."""
    url = 'https://www.sec.gov/files/company_tickers.json'
    req = urllib.request.Request(url, headers={'User-Agent': 'earnings-backtest/1.0 contact@example.com'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = _json.loads(resp.read().decode())
    return sorted({v['ticker'] for v in data.values()})

print('Fetching S&P 500 tickers from SPDR SPY holdings...')
TICKERS = get_sp500_tickers()

print(f'Backtest Configuration:')
print(f'  Period:           {START_DATE} to {END_DATE} ({BACKTEST_YEARS} years)')
print(f'  Target Delta:     {TARGET_DELTA} (+/- {DELTA_TOLERANCE})')
print(f'  Strikes Apart:    {STRIKES_APART}')
print(f'  Min Credit Ratio: {MIN_CREDIT_RATIO:.0%} of width')
print(f'  Universe:         {len(TICKERS)} tickers (S&P 500)')

Fetching S&P 500 tickers from SPDR SPY holdings...
Backtest Configuration:
  Period:           2021-05-11 to 2026-05-10 (5 years)
  Target Delta:     0.2 (+/- 0.05)
  Strikes Apart:    2
  Min Credit Ratio: 50% of width
  Universe:         501 tickers (S&P 500)


## Black-Scholes Pricing

In [70]:
def bs_price(S: float, K: float, T: float, r: float, sigma: float, 
             opt_type: str = 'put') -> float:
    """
    Calculate Black-Scholes option price.
    
    Parameters:
    -----------
    S : float - Spot price
    K : float - Strike price
    T : float - Time to expiration (years)
    r : float - Risk-free rate
    sigma : float - Volatility
    opt_type : str - 'call' or 'put'
    
    Returns:
    --------
    float - Option price
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if opt_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:  # put
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    return max(price, 0.0)


def bs_delta(S: float, K: float, T: float, r: float, sigma: float,
             opt_type: str = 'put') -> float:
    """
    Calculate Black-Scholes delta.
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    
    if opt_type == 'call':
        return norm.cdf(d1)
    else:  # put
        return norm.cdf(d1) - 1


def find_strike_for_delta(S: float, T: float, r: float, sigma: float,
                          target_delta: float, opt_type: str = 'put',
                          strike_increment: float = 1.0) -> float:
    """
    Find the strike price that gives approximately the target delta.
    Uses a simple search over standard strike increments.
    """
    # Generate potential strikes
    if opt_type == 'put':
        # For OTM puts, strikes are below spot
        strikes = np.arange(S * 0.70, S * 1.0, strike_increment)
        target = -abs(target_delta)  # Put deltas are negative
    else:
        # For OTM calls, strikes are above spot
        strikes = np.arange(S * 1.0, S * 1.30, strike_increment)
        target = abs(target_delta)  # Call deltas are positive
    
    best_strike = None
    best_diff = float('inf')
    
    for K in strikes:
        delta = bs_delta(S, K, T, r, sigma, opt_type)
        diff = abs(delta - target)
        if diff < best_diff:
            best_diff = diff
            best_strike = K
    
    return best_strike


def get_strike_increment(spot_price: float) -> float:
    """
    Determine standard strike increment based on stock price.
    """
    if spot_price < 25:
        return 0.5
    elif spot_price < 50:
        return 1.0
    elif spot_price < 200:
        return 2.5
    elif spot_price < 500:
        return 5.0
    else:
        return 10.0


# Test
print('Testing Black-Scholes functions...')
test_price = bs_price(S=100, K=95, T=7/365, r=0.03, sigma=0.35, opt_type='put')
test_delta = bs_delta(S=100, K=95, T=7/365, r=0.03, sigma=0.35, opt_type='put')
print(f'Put (S=100, K=95, 7 DTE, IV=35%): Price=${test_price:.2f}, Delta={test_delta:.3f}')

test_strike = find_strike_for_delta(S=100, T=7/365, r=0.03, sigma=0.35, target_delta=0.20, opt_type='put')
print(f'Strike for 0.20 delta put: ${test_strike:.0f}')
print('OK')

Testing Black-Scholes functions...
Put (S=100, K=95, 7 DTE, IV=35%): Price=$0.34, Delta=-0.137
Strike for 0.20 delta put: $96
OK


## Historical Earnings Data

In [71]:
def get_historical_earnings(ticker: str, start_date: date, end_date: date) -> pd.DataFrame:
    """
    Fetch historical earnings dates for a ticker.
    Uses yfinance earnings_dates which provides historical data.
    """
    try:
        t = yf.Ticker(ticker)
        
        # Get earnings dates (requires lxml)
        earnings = t.earnings_dates
        
        if earnings is None or earnings.empty:
            return pd.DataFrame()
        
        # Reset index to get dates as column
        earnings = earnings.reset_index()
        earnings.columns = ['earnings_datetime'] + list(earnings.columns[1:])
        
        # Convert to date
        earnings['earnings_date'] = pd.to_datetime(earnings['earnings_datetime']).dt.date
        
        # Filter to date range
        earnings = earnings[
            (earnings['earnings_date'] >= start_date) &
            (earnings['earnings_date'] <= end_date)
        ].copy()
        
        # Add day of week
        earnings['day_of_week'] = pd.to_datetime(earnings['earnings_date']).dt.day_name()
        earnings['is_friday'] = earnings['day_of_week'] == 'Friday'
        
        # Determine timing (BMO/AMC) from hour if available
        def get_timing(dt):
            if pd.isna(dt):
                return 'Unknown'
            try:
                hour = dt.hour
                if hour < 9 or (hour == 9 and dt.minute < 30):
                    return 'BMO'
                elif hour >= 16:
                    return 'AMC'
                else:
                    return 'During Market'
            except:
                return 'Unknown'
        
        earnings['timing'] = earnings['earnings_datetime'].apply(get_timing)
        
        earnings['ticker'] = ticker
        
        return earnings[['ticker', 'earnings_date', 'day_of_week', 'is_friday', 'timing']]
    
    except Exception as e:
        print(f'  [{ticker}] Error fetching earnings: {e}')
        return pd.DataFrame()


def get_next_earnings_info(ticker_obj) -> Tuple[Optional[date], str]:
    """
    Get the next earnings date and timing for current strategy scanning.
    Handles both dict and DataFrame formats from yfinance.
    """
    try:
        cal = ticker_obj.calendar
        
        if cal is None:
            return None, 'Unknown'
        
        # Handle both dict and DataFrame formats
        if isinstance(cal, dict):
            if 'Earnings Date' not in cal:
                return None, 'Unknown'
            raw = cal['Earnings Date']
            if isinstance(raw, (list, tuple)) and len(raw) > 0:
                raw = raw[0]
        elif isinstance(cal, pd.DataFrame):
            if cal.empty or 'Earnings Date' not in cal.index:
                return None, 'Unknown'
            raw = cal.loc['Earnings Date']
            if isinstance(raw, pd.Series):
                raw = raw.iloc[0] if len(raw) > 0 else None
        else:
            return None, 'Unknown'
        
        if raw is None:
            return None, 'Unknown'
        
        # Parse the earnings date
        if isinstance(raw, datetime):
            earnings_date = raw.date()
            hour = raw.hour
        elif isinstance(raw, date):
            earnings_date = raw
            hour = None
        elif isinstance(raw, pd.Timestamp):
            earnings_date = raw.date()
            hour = raw.hour if not pd.isna(raw.hour) else None
        else:
            try:
                ts = pd.Timestamp(raw)
                earnings_date = ts.date()
                hour = ts.hour if ts.hour != 0 else None
            except:
                return None, 'Unknown'
        
        # Determine timing
        if hour is None:
            timing = 'Unknown'
        elif hour < 9 or (hour == 9 and getattr(raw, 'minute', 30) < 30):
            timing = 'BMO'
        elif hour >= 16:
            timing = 'AMC'
        else:
            timing = 'During Market'
        
        return earnings_date, timing
    except:
        return None, 'Unknown'


def filter_friday_earnings(earnings_df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to Friday earnings that are BMO or AMC.
    """
    if earnings_df.empty:
        return earnings_df
    
    filtered = earnings_df[
        (earnings_df['is_friday'] == True) &
        (earnings_df['timing'].isin(['BMO', 'AMC', 'Unknown']))  # Include Unknown as we can't always determine
    ].copy()
    
    return filtered


print('Historical earnings functions defined.')

Historical earnings functions defined.


In [72]:
# ── Fetch all historical earnings ─────────────────────────────────────────────

print(f'Fetching historical earnings for {len(TICKERS)} tickers...')
print(f'Period: {START_DATE} to {END_DATE}\n')

all_earnings = []

for ticker in TICKERS:
    print(f'  {ticker}...', end=' ')
    earnings = get_historical_earnings(ticker, START_DATE, END_DATE)
    
    if not earnings.empty:
        friday_earnings = filter_friday_earnings(earnings)
        all_earnings.append(friday_earnings)
        print(f'{len(earnings)} total, {len(friday_earnings)} Friday earnings')
    else:
        print('no data')
    
    time.sleep(0.3)

if all_earnings:
    earnings_df = pd.concat(all_earnings, ignore_index=True)
    print(f'\nTotal Friday earnings events: {len(earnings_df)}')
else:
    earnings_df = pd.DataFrame()
    print('\nNo earnings data found.')

Fetching historical earnings for 501 tickers...
Period: 2021-05-11 to 2026-05-10

  A... 20 total, 0 Friday earnings
  AAPL... 20 total, 0 Friday earnings
  ABBV... 20 total, 11 Friday earnings
  ABNB... 21 total, 1 Friday earnings
  ABT... 20 total, 0 Friday earnings
  ACGL...   [ACGL] Error fetching earnings: ['Earnings Date']
no data
  ACN... 20 total, 2 Friday earnings
  ADBE... 20 total, 0 Friday earnings
  ADI... 20 total, 0 Friday earnings
  ADM... 20 total, 0 Friday earnings
  ADP... 20 total, 0 Friday earnings
  ADSK... 20 total, 0 Friday earnings
  AEE... 20 total, 0 Friday earnings
  AEP... 20 total, 0 Friday earnings
  AES... 20 total, 2 Friday earnings
  AFL...   [AFL] Error fetching earnings: ['Earnings Date']
no data
  AIG... 20 total, 0 Friday earnings
  AIZ... 20 total, 0 Friday earnings
  AJG... 20 total, 0 Friday earnings
  AKAM... 20 total, 0 Friday earnings
  ALB... 20 total, 0 Friday earnings
  ALGN... 20 total, 0 Friday earnings
  ALL... 20 total, 0 Friday earnin

In [73]:
# ── Display earnings distribution ─────────────────────────────────────────────

if not earnings_df.empty:
    print('Friday Earnings Distribution by Ticker:')
    print(earnings_df.groupby('ticker').size().sort_values(ascending=False))
    print()
    print('Sample of Friday earnings events:')
    display(earnings_df.head(20))
else:
    print('No Friday earnings events found.')

Friday Earnings Distribution by Ticker:
ticker
CBOE    20
AON     20
SLB     20
CL      20
CHTR    20
        ..
FDS      1
FCX      1
EPAM     1
DVA      1
ZBH      1
Length: 135, dtype: int64

Sample of Friday earnings events:


,ticker,earnings_date,day_of_week,is_friday,timing
0,ABBV,2025-10-31,Friday,True,BMO
1,ABBV,2025-04-25,Friday,True,BMO
2,ABBV,2025-01-31,Friday,True,BMO
3,ABBV,2024-04-26,Friday,True,BMO
4,ABBV,2024-02-02,Friday,True,BMO
5,ABBV,2023-10-27,Friday,True,BMO
6,ABBV,2022-10-28,Friday,True,BMO
7,ABBV,2022-07-29,Friday,True,BMO
8,ABBV,2022-04-29,Friday,True,BMO
9,ABBV,2021-10-29,Friday,True,BMO


In [74]:
earnings_df.ticker.unique()

array(['ABBV', 'ABNB', 'ACN', 'AES', 'AON', 'APD', 'APO', 'ARES', 'AXP',
       'AZO', 'BAC', 'BAX', 'BEN', 'BIIB', 'BK', 'BLK', 'BMY', 'BR', 'C',
       'CAG', 'CAH', 'CAT', 'CBOE', 'CBRE', 'CCL', 'CEG', 'CFG', 'CHD',
       'CHTR', 'CI', 'CL', 'CNC', 'CNP', 'CRH', 'CTVA', 'CVX', 'D', 'DAL',
       'DASH', 'DD', 'DE', 'DRI', 'DUK', 'DVA', 'EL', 'EPAM', 'ETN',
       'EVRG', 'FAST', 'FCX', 'FDS', 'FIS', 'FITB', 'FRT', 'FTV', 'GD',
       'GPN', 'GS', 'GWW', 'HAL', 'HBAN', 'HCA', 'HON', 'HSY', 'IDXX',
       'IQV', 'IT', 'ITW', 'JBHT', 'JBL', 'JCI', 'JPM', 'KIM', 'KKR',
       'KMB', 'KR', 'LHX', 'LIN', 'LNT', 'LYB', 'MMM', 'MRNA', 'MS',
       'MTB', 'NCLH', 'NEE', 'NEM', 'NRG', 'NSC', 'PANW', 'PAYX', 'PEP',
       'PG', 'PGR', 'PLD', 'PM', 'PNC', 'PNW', 'PPL', 'PSX', 'RCL', 'REG',
       'REGN', 'RF', 'ROP', 'RVTY', 'SATS', 'SCHW', 'SLB', 'SPG', 'SRE',
       'STT', 'STX', 'STZ', 'SW', 'SWK', 'SYF', 'TFC', 'TRMB', 'TROW',
       'TRV', 'UNH', 'USB', 'VMC', 'VST', 'VZ', 'WAB', 'WAT', '

## Historical Price Data

In [75]:
def get_historical_prices(ticker: str, start_date: date, end_date: date) -> pd.DataFrame:
    """
    Fetch historical price data for a ticker.
    """
    try:
        t = yf.Ticker(ticker)
        hist = t.history(
            start=start_date - timedelta(days=60),  # Extra buffer for volatility calc
            end=end_date + timedelta(days=10)
        )
        
        if hist.empty:
            return pd.DataFrame()
        
        hist = hist.reset_index()
        hist['Date'] = pd.to_datetime(hist['Date']).dt.date
        hist['ticker'] = ticker
        
        # Calculate historical volatility (20-day rolling)
        hist['returns'] = np.log(hist['Close'] / hist['Close'].shift(1))
        hist['hist_vol_20d'] = hist['returns'].rolling(20).std() * np.sqrt(252)
        
        return hist[['ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'hist_vol_20d']]
    
    except Exception as e:
        print(f'  [{ticker}] Error fetching prices: {e}')
        return pd.DataFrame()


print('Price fetching function defined.')

Price fetching function defined.


In [76]:
all_prices

[]

In [77]:
# ── Fetch historical prices for all tickers ───────────────────────────────────

print(f'Fetching historical prices for {len(TICKERS)} tickers...\n')

all_prices = []

for ticker in TICKERS:
    print(f'  {ticker}...', end=' ')
    prices = get_historical_prices(ticker, START_DATE, END_DATE)
    
    if not prices.empty:
        all_prices.append(prices)
        print(f'{len(prices)} days')
    else:
        print('no data')
    
    time.sleep(0.3)

if all_prices:
    prices_df = pd.concat(all_prices, ignore_index=True)
    print(f'\nTotal price records: {len(prices_df)}')
else:
    prices_df = pd.DataFrame()
    print('\nNo price data found.')

Fetching historical prices for 501 tickers...

  A... 1296 days
  AAPL... 1296 days
  ABBV... 

Failed to get ticker 'ABBV' reason: Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$ABBV: possibly delisted; no timezone found


no data
  ABNB... 1296 days
  ABT... 1296 days
  ACGL... 1296 days
  ACN... 1296 days
  ADBE... 1296 days
  ADI... 

Failed to get ticker 'ADI' reason: Failed to perform, curl: (28) Operation timed out after 934253 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$ADI: possibly delisted; no timezone found


no data
  ADM... 

...

  ZBH... no data


Failed to get ticker 'ZBRA' reason: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$ZBRA: possibly delisted; no timezone found


  ZBRA... no data


Failed to get ticker 'ZTS' reason: Failed to perform, curl: (6) Could not resolve host: query2.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$ZTS: possibly delisted; no timezone found


  ZTS... no data

Total price records: 9072


## Backtest Engine

In [32]:
def calculate_spread_pnl(spot_entry: float, spot_expiry: float,
                        short_strike: float, long_strike: float,
                        credit_received: float, opt_type: str = 'put') -> dict:
    """
    Calculate P&L for a credit spread at expiration.
    
    For Put Credit Spread (Bull Put):
    - Short higher strike put, long lower strike put
    - Max profit = credit received (if spot >= short strike)
    - Max loss = strike width - credit (if spot <= long strike)
    
    For Call Credit Spread (Bear Call):
    - Short lower strike call, long higher strike call
    - Max profit = credit received (if spot <= short strike)
    - Max loss = strike width - credit (if spot >= long strike)
    """
    strike_width = abs(short_strike - long_strike)
    max_profit = credit_received
    max_loss = strike_width - credit_received
    
    if opt_type == 'put':
        # Put credit spread: short higher strike, long lower strike
        if spot_expiry >= short_strike:
            # Both puts expire worthless
            pnl = credit_received
            outcome = 'max_profit'
        elif spot_expiry <= long_strike:
            # Both puts ITM, max loss
            pnl = -max_loss
            outcome = 'max_loss'
        else:
            # Short put ITM, long put OTM
            short_put_value = short_strike - spot_expiry
            pnl = credit_received - short_put_value
            outcome = 'partial'
    else:  # call
        # Call credit spread: short lower strike, long higher strike
        if spot_expiry <= short_strike:
            # Both calls expire worthless
            pnl = credit_received
            outcome = 'max_profit'
        elif spot_expiry >= long_strike:
            # Both calls ITM, max loss
            pnl = -max_loss
            outcome = 'max_loss'
        else:
            # Short call ITM, long call OTM
            short_call_value = spot_expiry - short_strike
            pnl = credit_received - short_call_value
            outcome = 'partial'
    
    return {
        'pnl': pnl,
        'pnl_pct': pnl / max_loss * 100 if max_loss > 0 else 0,  # P&L as % of risk
        'outcome': outcome,
        'max_profit': max_profit,
        'max_loss': max_loss,
        'credit_received': credit_received,
        'strike_width': strike_width,
    }


# Test
print('Testing spread P&L calculation...')
test_result = calculate_spread_pnl(
    spot_entry=100, spot_expiry=98,
    short_strike=95, long_strike=90,
    credit_received=2.50, opt_type='put'
)
print(f'Put spread (short 95, long 90, credit $2.50, spot at expiry $98):')
print(f'  P&L: ${test_result["pnl"]:.2f}, Outcome: {test_result["outcome"]}')
print('OK')

Testing spread P&L calculation...
Put spread (short 95, long 90, credit $2.50, spot at expiry $98):
  P&L: $2.50, Outcome: max_profit
OK


In [ ]:
def backtest_single_event(ticker: str, earnings_date: date, 
                        prices_df: pd.DataFrame,
                        spread_type: str = 'put') -> Optional[dict]:
    """
    Backtest a single earnings event.
    
    Parameters:
    -----------
    ticker : str - Stock ticker
    earnings_date : date - Earnings announcement date (Friday)
    prices_df : DataFrame - Historical prices
    spread_type : str - 'put' for bull put spread, 'call' for bear call spread
    
    Returns:
    --------
    dict with trade details and P&L, or None if trade not viable
    """
    try:
        # Get prices for this ticker
        ticker_prices = prices_df[prices_df['ticker'] == ticker].copy()
        if ticker_prices.empty:
            return None
        
        # Entry: Monday of earnings week (or first trading day)
        # Earnings is on Friday, so Monday is 4 days before
        entry_date = earnings_date - timedelta(days=4)
        
        # Find closest trading day to entry date
        entry_prices = ticker_prices[ticker_prices['Date'] <= entry_date].tail(1)
        if entry_prices.empty:
            return None
        
        entry_row = entry_prices.iloc[0]
        spot_entry = entry_row['Close']
        hist_vol = entry_row['hist_vol_20d']
        
        if pd.isna(hist_vol) or hist_vol <= 0:
            hist_vol = 0.30  # Default 30% vol
        
        # Adjust IV for earnings (typically elevated)
        implied_vol = hist_vol * EARNINGS_IV_MULTIPLIER
        
        # Time to expiration (Friday expiry, entering Monday = ~4 days)
        days_to_expiry = 5  # Simplified
        T = days_to_expiry / 365
        
        # Find strikes
        strike_inc = get_strike_increment(spot_entry)
        
        # Short leg: target delta
        short_strike = find_strike_for_delta(
            S=spot_entry, T=T, r=RISK_FREE_RATE, sigma=implied_vol,
            target_delta=TARGET_DELTA, opt_type=spread_type,
            strike_increment=strike_inc
        )
        
        if short_strike is None:
            return None
        
        # Round to nearest strike
        short_strike = round(short_strike / strike_inc) * strike_inc
        
        # Long leg: 2 strikes further OTM
        if spread_type == 'put':
            long_strike = short_strike - (STRIKES_APART * strike_inc)
        else:  # call
            long_strike = short_strike + (STRIKES_APART * strike_inc)
        
        strike_width = abs(short_strike - long_strike)
        
        # Calculate option prices
        short_price = bs_price(spot_entry, short_strike, T, RISK_FREE_RATE, implied_vol, spread_type)
        long_price = bs_price(spot_entry, long_strike, T, RISK_FREE_RATE, implied_vol, spread_type)
        
        credit_received = short_price - long_price
        
        if credit_received <= 0:
            return None
        
        # Check risk-neutral requirement: credit >= risk
        risk = strike_width - credit_received
        credit_ratio = credit_received / strike_width if strike_width > 0 else 0
        
        if credit_ratio < MIN_CREDIT_RATIO:
            return None  # Doesn't meet risk-neutral criteria
        
        # Verify delta is in range
        actual_delta = abs(bs_delta(spot_entry, short_strike, T, RISK_FREE_RATE, implied_vol, spread_type))
        if not (TARGET_DELTA - DELTA_TOLERANCE <= actual_delta <= TARGET_DELTA + DELTA_TOLERANCE):
            return None
        
        # Get expiry price (Friday close, or next trading day)
        expiry_prices = ticker_prices[ticker_prices['Date'] >= earnings_date].head(1)
        if expiry_prices.empty:
            # Try to get the last available price
            expiry_prices = ticker_prices[ticker_prices['Date'] <= earnings_date].tail(1)
        
        if expiry_prices.empty:
            return None
        
        spot_expiry = expiry_prices.iloc[0]['Close']
        expiry_date = expiry_prices.iloc[0]['Date']
        
        # Calculate P&L
        pnl_result = calculate_spread_pnl(
            spot_entry=spot_entry,
            spot_expiry=spot_expiry,
            short_strike=short_strike,
            long_strike=long_strike,
            credit_received=credit_received,
            opt_type=spread_type
        )
        
        # Calculate stock move
        stock_move = (spot_expiry - spot_entry) / spot_entry * 100
        
        return {
            'ticker': ticker,
            'earnings_date': earnings_date,
            'entry_date': entry_row['Date'],
            'expiry_date': expiry_date,
            'spread_type': spread_type,
            'spot_entry': round(spot_entry, 2),
            'spot_expiry': round(spot_expiry, 2),
            'stock_move_pct': round(stock_move, 2),
            'short_strike': short_strike,
            'long_strike': long_strike,
            'strike_width': strike_width,
            'short_delta': round(actual_delta, 3),
            'implied_vol': round(implied_vol, 3),
            'credit_received': round(credit_received, 2),
            'credit_ratio': round(credit_ratio, 3),
            'max_profit': round(pnl_result['max_profit'], 2),
            'max_loss': round(pnl_result['max_loss'], 2),
            'pnl': round(pnl_result['pnl'], 2),
            'pnl_pct': round(pnl_result['pnl_pct'], 2),
            'outcome': pnl_result['outcome'],
            'is_winner': pnl_result['pnl'] > 0,
        }
    
    except Exception as e:
        return None


print('Single event backtest function defined.')

Single event backtest function defined.


In [48]:
len(prices_df)

0

In [34]:
# ── Run Full Backtest ─────────────────────────────────────────────────────────

if earnings_df.empty or prices_df.empty:
    print('Missing data - cannot run backtest.')
    backtest_results = pd.DataFrame()
else:
    print(f'Running backtest on {len(earnings_df)} Friday earnings events...\n')
    
    results = []
    
    for idx, row in earnings_df.iterrows():
        ticker = row['ticker']
        earnings_date = row['earnings_date']
        
        # Test both put and call spreads
        for spread_type in ['put', 'call']:
            result = backtest_single_event(
                ticker=ticker,
                earnings_date=earnings_date,
                prices_df=prices_df,
                spread_type=spread_type
            )
            
            if result is not None:
                results.append(result)
    
    if results:
        backtest_results = pd.DataFrame(results)
        print(f'Completed: {len(backtest_results)} trades met all criteria')
        print(f'  Put spreads: {len(backtest_results[backtest_results["spread_type"] == "put"])}')
        print(f'  Call spreads: {len(backtest_results[backtest_results["spread_type"] == "call"])}')
    else:
        backtest_results = pd.DataFrame()
        print('No trades met all criteria.')

Missing data - cannot run backtest.


## Backtest Results

In [35]:
# ── Display Results ───────────────────────────────────────────────────────────

if not backtest_results.empty:
    print('Sample of backtest trades:')
    display_cols = [
        'ticker', 'earnings_date', 'spread_type',
        'spot_entry', 'spot_expiry', 'stock_move_pct',
        'short_strike', 'long_strike', 'short_delta',
        'credit_received', 'credit_ratio',
        'pnl', 'pnl_pct', 'outcome'
    ]
    display(backtest_results[display_cols].head(30))
else:
    print('No backtest results to display.')

No backtest results to display.


In [36]:
# ── Summary Statistics ────────────────────────────────────────────────────────

if not backtest_results.empty:
    print('=' * 70)
    print('BACKTEST SUMMARY')
    print('=' * 70)
    
    total_trades = len(backtest_results)
    winners = backtest_results['is_winner'].sum()
    losers = total_trades - winners
    win_rate = winners / total_trades * 100
    
    total_pnl = backtest_results['pnl'].sum()
    avg_pnl = backtest_results['pnl'].mean()
    avg_winner = backtest_results[backtest_results['is_winner']]['pnl'].mean() if winners > 0 else 0
    avg_loser = backtest_results[~backtest_results['is_winner']]['pnl'].mean() if losers > 0 else 0
    
    max_profit_trade = backtest_results['pnl'].max()
    max_loss_trade = backtest_results['pnl'].min()
    
    avg_credit = backtest_results['credit_received'].mean()
    avg_credit_ratio = backtest_results['credit_ratio'].mean()
    
    # Outcome distribution
    outcome_dist = backtest_results['outcome'].value_counts()
    
    print(f'\nTotal Trades:     {total_trades}')
    print(f'Winners:          {winners} ({win_rate:.1f}%)')
    print(f'Losers:           {losers} ({100-win_rate:.1f}%)')
    print()
    print(f'Total P&L:        ${total_pnl:,.2f}')
    print(f'Average P&L:      ${avg_pnl:.2f}')
    print(f'Average Winner:   ${avg_winner:.2f}')
    print(f'Average Loser:    ${avg_loser:.2f}')
    print(f'Max Profit:       ${max_profit_trade:.2f}')
    print(f'Max Loss:         ${max_loss_trade:.2f}')
    print()
    print(f'Avg Credit:       ${avg_credit:.2f}')
    print(f'Avg Credit Ratio: {avg_credit_ratio:.1%}')
    print()
    print('Outcome Distribution:')
    for outcome, count in outcome_dist.items():
        print(f'  {outcome}: {count} ({count/total_trades*100:.1f}%)')
    
    # By spread type
    print('\n' + '=' * 70)
    print('BY SPREAD TYPE')
    print('=' * 70)
    
    for spread_type in ['put', 'call']:
        subset = backtest_results[backtest_results['spread_type'] == spread_type]
        if len(subset) > 0:
            st_winners = subset['is_winner'].sum()
            st_win_rate = st_winners / len(subset) * 100
            st_total_pnl = subset['pnl'].sum()
            st_avg_pnl = subset['pnl'].mean()
            
            print(f'\n{spread_type.upper()} CREDIT SPREAD ({"Bullish" if spread_type == "put" else "Bearish"}):')
            print(f'  Trades:      {len(subset)}')
            print(f'  Win Rate:    {st_win_rate:.1f}%')
            print(f'  Total P&L:   ${st_total_pnl:,.2f}')
            print(f'  Avg P&L:     ${st_avg_pnl:.2f}')
else:
    print('No results to summarize.')

No results to summarize.


In [37]:
# ── Results by Ticker ─────────────────────────────────────────────────────────

if not backtest_results.empty:
    print('Performance by Ticker:')
    print()
    
    ticker_summary = (
        backtest_results
        .groupby('ticker')
        .agg(
            trades=('pnl', 'count'),
            winners=('is_winner', 'sum'),
            total_pnl=('pnl', 'sum'),
            avg_pnl=('pnl', 'mean'),
            avg_credit_ratio=('credit_ratio', 'mean'),
        )
        .round(2)
    )
    ticker_summary['win_rate'] = (ticker_summary['winners'] / ticker_summary['trades'] * 100).round(1)
    ticker_summary = ticker_summary.sort_values('total_pnl', ascending=False)
    
    display(ticker_summary)

In [38]:
# ── Results by Year ───────────────────────────────────────────────────────────

if not backtest_results.empty:
    backtest_results['year'] = pd.to_datetime(backtest_results['earnings_date']).dt.year
    
    print('Performance by Year:')
    print()
    
    year_summary = (
        backtest_results
        .groupby('year')
        .agg(
            trades=('pnl', 'count'),
            winners=('is_winner', 'sum'),
            total_pnl=('pnl', 'sum'),
            avg_pnl=('pnl', 'mean'),
        )
        .round(2)
    )
    year_summary['win_rate'] = (year_summary['winners'] / year_summary['trades'] * 100).round(1)
    
    display(year_summary)

## Visualizations

In [39]:
# ── Cumulative P&L Chart ──────────────────────────────────────────────────────

if not backtest_results.empty:
    # Sort by date and calculate cumulative P&L
    results_sorted = backtest_results.sort_values('earnings_date').copy()
    results_sorted['cumulative_pnl'] = results_sorted['pnl'].cumsum()
    results_sorted['trade_num'] = range(1, len(results_sorted) + 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Cumulative P&L
    ax1 = axes[0, 0]
    ax1.plot(results_sorted['trade_num'], results_sorted['cumulative_pnl'], 
             linewidth=2, color='blue')
    ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax1.fill_between(results_sorted['trade_num'], 0, results_sorted['cumulative_pnl'],
                     where=results_sorted['cumulative_pnl'] >= 0, alpha=0.3, color='green')
    ax1.fill_between(results_sorted['trade_num'], 0, results_sorted['cumulative_pnl'],
                     where=results_sorted['cumulative_pnl'] < 0, alpha=0.3, color='red')
    ax1.set_xlabel('Trade Number')
    ax1.set_ylabel('Cumulative P&L ($)')
    ax1.set_title('Cumulative P&L Over Time')
    ax1.grid(True, alpha=0.3)
    
    # 2. P&L Distribution
    ax2 = axes[0, 1]
    ax2.hist(backtest_results['pnl'], bins=30, edgecolor='black', alpha=0.7)
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax2.axvline(x=backtest_results['pnl'].mean(), color='green', linestyle='-', 
                linewidth=2, label=f'Mean: ${backtest_results["pnl"].mean():.2f}')
    ax2.set_xlabel('P&L ($)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('P&L Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Win Rate by Spread Type
    ax3 = axes[1, 0]
    spread_stats = backtest_results.groupby('spread_type').agg(
        win_rate=('is_winner', 'mean'),
        count=('is_winner', 'count')
    )
    colors = ['green' if x > 0.5 else 'red' for x in spread_stats['win_rate']]
    bars = ax3.bar(spread_stats.index, spread_stats['win_rate'] * 100, color=colors, alpha=0.7)
    ax3.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Spread Type')
    ax3.set_ylabel('Win Rate (%)')
    ax3.set_title('Win Rate by Spread Type')
    ax3.set_ylim(0, 100)
    for bar, count in zip(bars, spread_stats['count']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                 f'n={count}', ha='center', va='bottom')
    ax3.grid(True, alpha=0.3)
    
    # 4. Outcome Distribution
    ax4 = axes[1, 1]
    outcome_counts = backtest_results['outcome'].value_counts()
    colors_outcome = {'max_profit': 'green', 'partial': 'orange', 'max_loss': 'red'}
    ax4.pie(outcome_counts, labels=outcome_counts.index, autopct='%1.1f%%',
            colors=[colors_outcome.get(x, 'gray') for x in outcome_counts.index])
    ax4.set_title('Outcome Distribution')
    
    plt.tight_layout()
    plt.show()
else:
    print('No data to visualize.')

No data to visualize.


In [40]:
# ── P&L vs Stock Move ─────────────────────────────────────────────────────────

if not backtest_results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, spread_type in enumerate(['put', 'call']):
        ax = axes[idx]
        subset = backtest_results[backtest_results['spread_type'] == spread_type]
        
        if len(subset) > 0:
            colors = ['green' if x else 'red' for x in subset['is_winner']]
            ax.scatter(subset['stock_move_pct'], subset['pnl'], c=colors, alpha=0.6, s=50)
            ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
            ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
            ax.set_xlabel('Stock Move (%)')
            ax.set_ylabel('P&L ($)')
            ax.set_title(f'{spread_type.upper()} Spread: P&L vs Stock Move')
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [41]:
# ── Yearly Performance Chart ──────────────────────────────────────────────────

if not backtest_results.empty and 'year' in backtest_results.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    year_data = backtest_results.groupby('year').agg(
        total_pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        win_rate=('is_winner', 'mean')
    )
    
    # Total P&L by year
    ax1 = axes[0]
    colors = ['green' if x > 0 else 'red' for x in year_data['total_pnl']]
    ax1.bar(year_data.index.astype(str), year_data['total_pnl'], color=colors, alpha=0.7)
    ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Total P&L ($)')
    ax1.set_title('Total P&L by Year')
    ax1.grid(True, alpha=0.3)
    
    # Win rate by year
    ax2 = axes[1]
    colors = ['green' if x > 0.5 else 'red' for x in year_data['win_rate']]
    ax2.bar(year_data.index.astype(str), year_data['win_rate'] * 100, color=colors, alpha=0.7)
    ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Win Rate (%)')
    ax2.set_title('Win Rate by Year')
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Risk Metrics

In [42]:
# ── Risk Metrics ──────────────────────────────────────────────────────────────

if not backtest_results.empty:
    print('=' * 70)
    print('RISK METRICS')
    print('=' * 70)
    
    # Sort by date for drawdown calculation
    results_sorted = backtest_results.sort_values('earnings_date').copy()
    results_sorted['cumulative_pnl'] = results_sorted['pnl'].cumsum()
    
    # Maximum drawdown
    results_sorted['peak'] = results_sorted['cumulative_pnl'].cummax()
    results_sorted['drawdown'] = results_sorted['cumulative_pnl'] - results_sorted['peak']
    max_drawdown = results_sorted['drawdown'].min()
    
    # Profit factor
    gross_profit = backtest_results[backtest_results['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(backtest_results[backtest_results['pnl'] < 0]['pnl'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Sharpe-like ratio (using trade returns)
    avg_return = backtest_results['pnl_pct'].mean()
    std_return = backtest_results['pnl_pct'].std()
    sharpe_like = avg_return / std_return if std_return > 0 else 0
    
    # Consecutive wins/losses
    results_sorted['is_winner_int'] = results_sorted['is_winner'].astype(int)
    
    def max_consecutive(series, value):
        groups = (series != value).cumsum()
        return series.groupby(groups).sum().max()
    
    max_consec_wins = max_consecutive(results_sorted['is_winner_int'], 0)
    max_consec_losses = max_consecutive(1 - results_sorted['is_winner_int'], 0)
    
    # Average risk per trade
    avg_risk = backtest_results['max_loss'].mean()
    
    print(f'\nMax Drawdown:          ${max_drawdown:.2f}')
    print(f'Profit Factor:         {profit_factor:.2f}')
    print(f'Sharpe-like Ratio:     {sharpe_like:.2f}')
    print(f'Max Consecutive Wins:  {int(max_consec_wins)}')
    print(f'Max Consecutive Losses:{int(max_consec_losses)}')
    print(f'Average Risk/Trade:    ${avg_risk:.2f}')
    print(f'Gross Profit:          ${gross_profit:,.2f}')
    print(f'Gross Loss:            ${gross_loss:,.2f}')
    
    # Risk-adjusted return
    total_pnl = backtest_results['pnl'].sum()
    total_risk = backtest_results['max_loss'].sum()
    risk_adjusted_return = total_pnl / total_risk * 100 if total_risk > 0 else 0
    
    print(f'\nRisk-Adjusted Return:  {risk_adjusted_return:.2f}%')
    print(f'  (Total P&L / Total Max Risk)')

## Export Results

In [43]:
# ── Export to CSV ─────────────────────────────────────────────────────────────

if not backtest_results.empty:
    output_filename = f'earnings_spread_backtest_{END_DATE.isoformat()}.csv'
    backtest_results.to_csv(output_filename, index=False)
    print(f'Exported {len(backtest_results)} trades to: {output_filename}')
else:
    print('No data to export.')

No data to export.


## Conclusions & Notes

### Strategy Summary
- **Entry**: Monday of earnings week (earnings on Friday)
- **Short Leg**: ~0.20 delta, 2 strikes closer to ATM than long leg
- **Risk-Neutral**: Credit received >= 50% of strike width
- **Exit**: Hold to expiration

### Limitations of This Backtest
1. **Estimated Options Prices**: Uses Black-Scholes with estimated IV, not actual market prices
2. **IV Estimation**: Historical volatility * 1.5x multiplier is a rough approximation
3. **No Bid-Ask Spread**: Assumes mid-price execution (unrealistic)
4. **No Early Exit**: Holds to expiration only
5. **Simplified Expiration**: Assumes Friday close price for P&L

### For More Accurate Backtesting
Consider using:
- **OptionMetrics**: Academic-grade historical options data
- **CBOE DataShop**: Official exchange data
- **ORATS**: Professional options analytics with historical data
- **ThetaData**: Cost-effective historical options data

### Key Metrics to Watch
- **Win Rate**: Should be >50% for risk-neutral strategy
- **Profit Factor**: >1.0 means profitable
- **Max Drawdown**: Understand worst-case scenario
- **Consistency**: Year-over-year performance